<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، مدل embedding، دیتاست قرآن و بردارهای ذخیره‌شده بارگذاری می‌شوند تا پایپ‌لاین RAG مستقل از Notebook قبلی آماده شود
</div>

In [1]:
# Load retrieval resources

from pathlib import Path
import json
import numpy as np
from sentence_transformers import SentenceTransformer

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

dataset_file = project_root / "data" / "processed" / "quran_dataset_clean.json"
embeddings_file = project_root / "data" / "embeddings" / "quran_embeddings.npy"

with open(dataset_file, "r", encoding="utf-8") as f:
    quran_data = json.load(f)

quran_embeddings = np.load(embeddings_file)

model_name = "intfloat/multilingual-e5-small"
embedding_model = SentenceTransformer(model_name, device="cpu")

print("Records:", len(quran_data))
print("Embeddings:", quran_embeddings.shape)
print("Model:", model_name)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Records: 6236
Embeddings: (6236, 384)
Model: intfloat/multilingual-e5-small


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، یک تابع بازیابی ساخته می‌شود که سؤال کاربر را به embedding تبدیل می‌کند و مرتبط‌ترین آیات را همراه با متن عربی، ترجمه‌ها و امتیاز شباهت برمی‌گرداند.
</div>

In [2]:
# Retrieve relevant Quran verses

def retrieve_quran_verses(query, top_k=5):
    """
    Retrieve the most relevant Quran verses for a user query
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    top_k = min(top_k, len(quran_data))

    query_embedding = embedding_model.encode(
        [f"query: {query.strip()}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]

    scores = quran_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = quran_data[index]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "surah": item["surah"],
            "ayah": item["ayah"],
            "arabic": item["arabic"],
            "fooladvand": item["fooladvand"],
            "ansarian": item["ansarian"],
        })

    return results


print("Quran retrieval function is ready.")

Quran retrieval function is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، آیات بازیابی‌شده به یک متن ساختاریافته تبدیل می‌شوند تا بعداً به مدل زبانی داده شوند و پاسخ فقط بر اساس همین آیات ساخته شود
</div>

In [3]:
# Build structured Quran context

def build_quran_context(results):
    """
    Convert retrieved Quran verses into structured context
    """
    if not results:
        return ""

    context_parts = []

    for result in results:
        verse_context = (
            f"[سوره {result['surah']}، آیه {result['ayah']}]\n"
            f"متن عربی: {result['arabic']}\n"
            f"ترجمه فولادوند: {result['fooladvand']}\n"
            f"ترجمه انصاریان: {result['ansarian']}"
        )

        context_parts.append(verse_context)

    return "\n\n".join(context_parts)


print("Quran context builder is ready.")

Quran context builder is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، ابتدا آیات مرتبط بازیابی می‌شوند و سپس خروجی آن‌ها به context ساختاریافته تبدیل و نمایش داده می‌شود.
</div>

In [4]:
# Test retrieval and context building

query = "خدا توبه کنندگان را دوست دارد"

retrieved_verses = retrieve_quran_verses(query, top_k=3)
quran_context = build_quran_context(retrieved_verses)

print("Query:")
print(query)

print("\nQuran Context:")
print(quran_context)

Query:
خدا توبه کنندگان را دوست دارد

Quran Context:
[سوره 40، آیه 7]
متن عربی: ٱلَّذِينَ يَحْمِلُونَ ٱلْعَرْشَ وَمَنْ حَوْلَهُۥ يُسَبِّحُونَ بِحَمْدِ رَبِّهِمْ وَيُؤْمِنُونَ بِهِۦ وَيَسْتَغْفِرُونَ لِلَّذِينَ ءَامَنُوا۟ رَبَّنَا وَسِعْتَ كُلَّ شَىْءٍ رَّحْمَةً وَعِلْمًا فَٱغْفِرْ لِلَّذِينَ تَابُوا۟ وَٱتَّبَعُوا۟ سَبِيلَكَ وَقِهِمْ عَذَابَ ٱلْجَحِيمِ
ترجمه فولادوند: کسانی که عرش [خدا] را حمل می‌کنند، و آنها که پیرامون آنند، به سپاس پروردگارشان تسبیح می‌گویند و به او ایمان دارند و برای کسانی که گرویده‌اند طلب آمرزش می‌کنند: «پروردگارا، رحمت و دانش [تو بر] هر چیز احاطه دارد؛ کسانی را که توبه کرده و راه تو را دنبال کرده‌اند ببخش و آنها را از عذاب آتش نگاه دار.»
ترجمه انصاریان: فرشتگانی که عرش را حمل می کنند و آنان که پیرامون آن هستند، همراه سپاس و ستایش، پروردگارشان را تسبیح می گویند و به او ایمان دارند و برای اهل ایمان آمرزش می طلبند، [و می گویند:] پروردگارا! از روی رحمت و دانش همه چیز را فرا گرفته ای، پس آنان را که توبه کرده اند و راه تو را پیروی نموده اند بیامرز، و آنان را از عذاب دوز

<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، سؤال کاربر و context آیات در یک prompt قرار می‌گیرند تا مدل فقط بر اساس همین آیات پاسخ دهد و از اطلاعات بیرونی استفاده نکند.
</div>

In [5]:
# Build a grounded RAG prompt

def build_rag_prompt(query, quran_context):
    """
    Build a prompt that restricts the answer to the retrieved Quran verses
    """
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string.")

    if not isinstance(quran_context, str) or not quran_context.strip():
        raise ValueError("Quran context must be a non-empty string.")

    prompt = f"""
شما یک دستیار پاسخ‌گو بر اساس قرآن هستید.

قوانین:
1. فقط بر اساس آیات و ترجمه‌های موجود در بخش «متن مرجع» پاسخ بده.
2. از تفسیر، حدیث، اطلاعات تاریخی یا دانش بیرونی استفاده نکن.
3. اگر پاسخ مستقیم در متن مرجع وجود ندارد، صریح بگو:
   «در آیات بازیابی‌شده، پاسخ مستقیم و کافی پیدا نشد.»
4. پاسخ را کوتاه، روشن و دقیق بنویس.
5. در پایان، شماره سوره و آیه‌های استفاده‌شده را ذکر کن.
6. هیچ مطلبی را به قرآن نسبت نده مگر اینکه در متن مرجع وجود داشته باشد.

سؤال کاربر:
{query}

متن مرجع:
{quran_context}

پاسخ:
""".strip()

    return prompt


print("RAG prompt builder is ready.")

RAG prompt builder is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، prompt نهایی RAG با استفاده از سؤال کاربر و context آیات ساخته و نمایش داده می‌شود تا قبل از اتصال به مدل زبانی بررسی شود.
</div>

In [6]:
# Test the RAG prompt builder

rag_prompt = build_rag_prompt(
    query=query,
    quran_context=quran_context
)

print(rag_prompt)

شما یک دستیار پاسخ‌گو بر اساس قرآن هستید.

قوانین:
1. فقط بر اساس آیات و ترجمه‌های موجود در بخش «متن مرجع» پاسخ بده.
2. از تفسیر، حدیث، اطلاعات تاریخی یا دانش بیرونی استفاده نکن.
3. اگر پاسخ مستقیم در متن مرجع وجود ندارد، صریح بگو:
   «در آیات بازیابی‌شده، پاسخ مستقیم و کافی پیدا نشد.»
4. پاسخ را کوتاه، روشن و دقیق بنویس.
5. در پایان، شماره سوره و آیه‌های استفاده‌شده را ذکر کن.
6. هیچ مطلبی را به قرآن نسبت نده مگر اینکه در متن مرجع وجود داشته باشد.

سؤال کاربر:
خدا توبه کنندگان را دوست دارد

متن مرجع:
[سوره 40، آیه 7]
متن عربی: ٱلَّذِينَ يَحْمِلُونَ ٱلْعَرْشَ وَمَنْ حَوْلَهُۥ يُسَبِّحُونَ بِحَمْدِ رَبِّهِمْ وَيُؤْمِنُونَ بِهِۦ وَيَسْتَغْفِرُونَ لِلَّذِينَ ءَامَنُوا۟ رَبَّنَا وَسِعْتَ كُلَّ شَىْءٍ رَّحْمَةً وَعِلْمًا فَٱغْفِرْ لِلَّذِينَ تَابُوا۟ وَٱتَّبَعُوا۟ سَبِيلَكَ وَقِهِمْ عَذَابَ ٱلْجَحِيمِ
ترجمه فولادوند: کسانی که عرش [خدا] را حمل می‌کنند، و آنها که پیرامون آنند، به سپاس پروردگارشان تسبیح می‌گویند و به او ایمان دارند و برای کسانی که گرویده‌اند طلب آمرزش می‌کنند: «پروردگارا، رحمت

<div style="direction: rtl; white-space: normal; line-height: 1;">
تنظیمات از فایل .env خوانده می‌شوند و یک درخواست آزمایشی به مدل Qwen ارسال می‌شود. کلید API در خروجی نمایش داده نمی‌شود.
</div>

In [7]:
# Test Qwen API connection

from pathlib import Path
import os
import json
import urllib.request
import urllib.error


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load_env_file(env_path):
    """
    Load environment variables from a .env file
    """
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#") or "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


env_file = project_root / ".env"
load_env_file(env_file)

api_key = os.getenv("ARVAN_AI_API_KEY")
chat_url = os.getenv("ARVAN_CHAT_URL")
chat_model = os.getenv("ARVAN_CHAT_MODEL")

if not api_key:
    raise ValueError("ARVAN_AI_API_KEY is missing.")

if not chat_url:
    raise ValueError("ARVAN_CHAT_URL is missing.")

if not chat_model:
    raise ValueError("ARVAN_CHAT_MODEL is missing.")

payload = {
    "model": chat_model,
    "messages": [
        {
            "role": "user",
            "content": "فقط با یک جمله کوتاه پاسخ بده: اتصال برقرار است."
        }
    ],
    "temperature": 0,
    "max_tokens": 50
}

request = urllib.request.Request(
    chat_url,
    data=json.dumps(payload).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    method="POST"
)

try:
    with urllib.request.urlopen(request, timeout=60) as response:
        result = json.loads(response.read().decode("utf-8"))

    answer = result["choices"][0]["message"]["content"]

    print("Model:", chat_model)
    print("Connection: successful")
    print("Response:", answer)

except urllib.error.HTTPError as error:
    error_body = error.read().decode("utf-8")
    print("HTTP status:", error.code)
    print("Error response:", error_body)

except urllib.error.URLError as error:
    print("Connection error:", error.reason)

HTTP status: 404
Error response: 404 page not found



<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، تنظیمات Qwen از فایل .env خوانده می‌شود و اتصال مدل با همان ساختار OpenAI-compatible پروژه قبلی آزمایش می‌شود.
</div>

In [8]:
# Test Qwen connection with OpenAI-compatible client

import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

load_dotenv(project_root / ".env")

api_key = os.getenv("ARVAN_CHAT_API_KEY") or os.getenv("ARVAN_AI_API_KEY")
base_url = os.getenv("ARVAN_CHAT_URL")
chat_model = os.getenv("ARVAN_CHAT_MODEL") or "Qwen3-30B-A3B"

if not api_key:
    raise ValueError("ARVAN_AI_API_KEY is missing.")

if not base_url:
    raise ValueError("ARVAN_CHAT_URL is missing.")

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {
            "role": "user",
            "content": "فقط بنویس: اتصال برقرار است."
        }
    ],
    temperature=0,
    max_tokens=30
)

print("Model:", chat_model)
print("Connection: successful")
print("Response:", response.choices[0].message.content.strip())

Model: Qwen3-30B-A3B
Connection: successful
Response: اتصال برقرار است.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، prompt ساخته‌شده برای RAG به مدل Qwen ارسال می‌شود تا پاسخ نهایی فقط بر اساس آیات بازیابی‌شده تولید شود.
</div>

In [9]:
# Generate a grounded answer with Qwen

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {
            "role": "user",
            "content": rag_prompt
        }
    ],
    temperature=0,
    max_tokens=500
)

generated_answer = response.choices[0].message.content.strip()

print("Generated Answer:")
print(generated_answer)

Generated Answer:
خداوند توبه کنندگان را دوست دارد.  
[سوره 2، آیه 222]


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، prompt نهایی از سؤال و context موجود ساخته می‌شود تا برای مدل Qwen آماده باشد.
</div>

In [10]:
# Build the final RAG prompt

rag_prompt = build_rag_prompt(
    query=query,
    quran_context=quran_context
)

print("RAG prompt is ready.")

RAG prompt is ready.
